# 04 · Bonus B: Mixture-of-Experts, inspect it, then fine-tune it
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/57Ajay/modeltune/blob/main/notebooks/04_bonus_moe_granite.ipynb)

Companion to [chapter 13](https://github.com/57Ajay/modeltune/blob/main/course/13_mixture_of_experts.md). We open up IBM Granite-3.1-3B-A800M (40 experts per
layer, 8 active per token), count *total* vs *active* parameters, watch the router pick experts, and then fine-tune
it on **exactly** Project 1's task so you can compare a dense 0.6B model with a 3.3B-total / 0.8B-active MoE.

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Any newer GPU works too; the code picks fp16 or bf16 by itself.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Get the course code, then install the pinned libraries from pyproject.toml with uv.
# Colab already ships a CUDA build of PyTorch; `uv pip install --system` keeps it instead of downloading another.
!test -d modeltune || git clone -q https://github.com/57Ajay/modeltune.git
%cd modeltune
!git pull -q
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q -r pyproject.toml

In [ ]:
# Hugging Face token: add HF_TOKEN in Colab's key icon (Secrets) panel, with WRITE access to push models.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import login
    login()
# Pushing to someone else's namespace fails: change hub.repo_id in the configs (or override it) to YOUR username.

## 1. Total vs active parameters, and what the router does

In [ ]:
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
mid = "ibm-granite/granite-3.1-3b-a800m-instruct"
tok = AutoTokenizer.from_pretrained(mid)
model = AutoModelForCausalLM.from_pretrained(mid, dtype=torch.float16, device_map={"": 0})
cfg = model.config
print(model.model.layers[0])
total = sum(p.numel() for p in model.parameters())
experts = sum(p.numel() for n, p in model.named_parameters() if ".experts." in n)
k, E = cfg.num_experts_per_tok, cfg.num_local_experts
active = total - experts + experts * k / E
print(f"total {total/1e9:.2f}B | in experts {experts/1e9:.2f}B | active per token ~{active/1e9:.2f}B ({k} of {E} experts)")

In [ ]:
text = "The European Central Bank raised interest rates in Frankfurt, while Lionel Messi scored twice for Inter Miami."
enc = tok(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model(**enc, output_router_logits=True)
for layer in (0, len(out.router_logits) // 2, len(out.router_logits) - 1):
    chosen = out.router_logits[layer].float().topk(k, dim=-1).indices.flatten().tolist()
    print(f"layer {layer:2d}: experts used by {enc.input_ids.shape[1]} tokens -> {sorted(Counter(chosen).items(), key=lambda x: -x[1])[:10]} ...")
del model; torch.cuda.empty_cache()

## 2. Baseline on the Project 1 task

In [ ]:
!python -m modeltune.evaluate configs/bonus_moe_granite_ner_json_lora.yaml --base

## 3. Fine-tune: LoRA on attention + experts, router frozen, load-balancing loss on

In [ ]:
!python -m modeltune.train configs/bonus_moe_granite_ner_json_lora.yaml

In [ ]:
!python -m modeltune.evaluate configs/bonus_moe_granite_ner_json_lora.yaml

## 4. Compare with Project 1 (dense Qwen3-0.6B)
If you ran notebook 01 in this runtime, its results are in `outputs/p1-qwen3-0.6b-ner-json-lora/`.

In [ ]:
import json, os, pandas as pd
cols = {}
for name, path in {"dense 0.6B (P1)": "outputs/p1-qwen3-0.6b-ner-json-lora/eval_finetuned.json",
                   "MoE 3.3B/0.8B base": "outputs/bonus-granite-moe-ner-json-lora/eval_base.json",
                   "MoE 3.3B/0.8B tuned": "outputs/bonus-granite-moe-ner-json-lora/eval_finetuned.json"}.items():
    if os.path.exists(path):
        cols[name] = json.load(open(path))["results"]
pd.DataFrame(cols).round(3)

## Experiments
1. **Attention-only LoRA:** delete `lora.target_parameters` → compare F1, trainable params and speed.
2. **No load balancing:** `train.router_aux_loss_coef=0.0`. Re-run the router cell on the fine-tuned model. Did expert usage get more skewed?
3. Why didn't we use 4-bit here? Try `model.quantization=4bit` and compare peak memory printed after training (chapter 13 explains the result).